## AI4Climate ML tutorial - Data Exploration
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered


![climate_zones](https://upload.wikimedia.org/wikipedia/commons/thumb/3/32/K%C3%B6ppen-Geiger_Climate_Classification_Map_%281980%E2%80%932016%29_no_borders.png/1920px-K%C3%B6ppen-Geiger_Climate_Classification_Map_%281980%E2%80%932016%29_no_borders.png) 
The climate of different areas of the world is classified into different types, based on the Koppen-Geiger scheme. As the climate changes, some areas may change the classification of their climate 
In this series of tutorial notebooks, we willlearn how to build a machine learning pipeline by building a model to predict the clinate zone of a location from key climate means and indicators. We will look at how climate models predict shifts in the classification of climate zones as the climate changes in the future.

Links
 - [Climate Zones- Wikipedia](https://en.wikipedia.org/wiki/K%C3%B6ppen_climate_classification)
 - [Climate Zones Dataset](https://www.gloh2o.org/koppen/#:~:text=The%20K%C3%B6ppen%2DGeiger%20climate%20classification%20maps%20are%20high%2Dresolution,Climate%20Sensitivity%20(ECS)%2C%20and%20historical%20warming%20trend) 
    

### Prerequisites 

To be able to sucessfully work through this notebook, you will need some understanding of the following
- The python programming language
- Simple data handling in python
- Basic plotting using matplotlib
- Environment set up using pip or conda

### Learning outcomes from completing the notebook
* Loading and exploring tabular and gridded data using pandas
* typical techniques for exploratory data analysis on different sorts of data
* key characteristics of datasets to use in choosing components for a machine learning pipeline

## Tutorial - Exploratory Data Analysis
The first step in a machine learning problem is to understand the data you have available for training a machine learning algorithm. In any data driven technique, the quality of the results is only going to be as good as the quality of the data. This can only be achieved by matching the appropriate techniques to the data that you have, so understanding the particular dataset to be used is the foundation of a successful outcome.

To achieve this, one typically performs an Exploratory Data Analysis, where one produce a series of summary statistics and plots that highlight the most salient characteristics of the dataset for the problem at hand. Typical traits to be considered include:

- The range of values different input feature can take e.g. min and max temperature, start and end dates.
- Important subsets of the data e.g. different seasons or time of the day.
- Distributions of different features, both as a whole and important subsets
- Correlations between different features, especially input and target features.



### Dataset 1 - Climate Zones

The first dataset we will be looking at is a **tabular dataset** of classification of the climate zones of land points on earth. Tabular dataset is one where the rows of the table represent different data points, for example observation at different times or different locations. The columns represent different attributes of the data point. This can include metadata such as the time and location for the collection of the data (for example a temperature measurement), or for which the data applies (for example of weather forecast). 

In this case each row in our dataset will represent a particular point on the land on earth for a particular climate time period. The different columns represent different attributes, usually referred to in data science as **features**. In this data set features will include:
* location specified by latitude and longtitude
* climate period
* scenario - either `historical` for past climate periods, or a [*shared socioeconomic pathway* or *SSP*](https://en.wikipedia.org/wiki/Shared_Socioeconomic_Pathways) for future periods based on climate projection.
* monthly means and standard deviations of temperature and precipitation  for each month of the year, averaged for the climate period.  

The original data stores the data in a different way, as a **gridded dataset**. Gridded data is where the data is not simply rows and columns, and different axes represent different dimensions of the data. Typically this will at a minimum be latitude and longitude, and also usually a time axis. In addition you might have data representing vertical levels of the atmosphere on another axis. This format is less common generally but is widely used for geospatial data such as used in earth sciences, including weather and climate research. Different features which are columns in the tabular data are often separate multi-dimensional arrays in gridded format.



### Import of libraries
In this tutorial we will use some standard python libraries for weather and climate research. In addtion to base python libraries we are using:
- matplotlib for plotting
- cartopy for handling projections of the data for plotting
- pandas for loading and manipulating tabular data
- xarray for loading and manipulating gridded data. 

In [1]:
import pathlib
import os
import json

In [2]:
import matplotlib.pyplot
import cartopy.crs

In [3]:
import xarray
import pandas

### Define data access parameters
In order to load and explore the dataset, we need to define some of the parameters of the data which we might use to access subsets of the data  of interest. Key metadata definiitions have been pulled out into a single JSON config file so we can have consistent definiition to use across all the notebooks in this tutorial.

In [4]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'jasmin',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/ssde/j25a/mmh_storage/ai4c_data/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer

In [5]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path


In [6]:
current_platform = tutorial_config['platform']
current_platform

'jasmin'

In [ ]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

In [ ]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

In [ ]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [ ]:
time_periods

In [ ]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
period_scenario_template = 'climate_zones_{start}_{end}_{scenario}_{res}.csv'
ml_ready_fname_template =  tutorial_config['csv_out_template']

In [ ]:
current_res = 1.0

In [ ]:
climate_subgroups_dict = tutorial_config['climate_subgroups']#
climate_subgroups_dict

In [ ]:
climate_subgroup_lookup = [k1 for k1 in climate_subgroups_dict.keys()]

In [ ]:
climate_subgroup_lookup

In [ ]:
climate_group_lookup = [k1[0] for k1 in climate_subgroups_dict.keys()]
climate_group_lookup[0] = 'none'
climate_group_lookup

### Load the data
We will now load data for historic and future periods in order to understand how the distributions change and use this to inform our model training.
We will start by loading the gridded data and exploring that, before we look at the preprocessed ML ready data which we will use for training a machine learning model.

In [ ]:
def get_data_path(root_dir, time_period, scenario_id, prefix, resolution_str, suffix) :
    start_year = time_period[0]
    end_year = time_period[1]
    if scenario_id == historic_scenario_str:
        data_dir = root_data_dir / time_dir_template.format(start_year=start_year,end_year=end_year)
    else:
        data_dir = root_data_dir / time_dir_template.format(start_year=start_year,end_year=end_year) / scenario_id 
    data_fname = fname_template.format(prefix=prefix, 
                                       res=resolution_str, 
                                       suffix=format_str)
    return data_dir / data_fname

In [ ]:
data_path_dict = {
    (start_year, end_year): {
        scenario_id: { ds_id: { current_res: get_data_path(root_dir=root_data_dir,
                                                           time_period=(start_year, end_year),
                                                           scenario_id=scenario_id, 
                                                           prefix=ds_str,
                                                           resolution_str=res_str,
                                                           suffix=format_str,
                                                          )
                                for current_res, res_str in resolutions_dict.items() 
                              }
                       for ds_id, ds_str in dataset_prefix_dict.items()
                     } 
        for scenario_id in current_scenarios
    }
    for (start_year, end_year), current_scenarios in time_periods.items()                                                                                                             
}

In [ ]:
data_path_dict.keys()

In [ ]:
data_path_dict[(1901,1930)].keys()

In [ ]:
data_path_dict[(2071,2099)].keys()

In [ ]:
data_path_dict[(1901,1930)]['historic'].keys()

In [ ]:
data_path_dict[(1901,1930)]['historic']['climate_mean'].keys()

In [ ]:
data_path_dict[(1901,1930)]['historic']['climate_mean'][0.1]

In [ ]:
select_historic = (1901,1930)
select_future = (2071,2099)
select_future_scenario = 'ssp370'

In [ ]:
data_dict = {
    current_period: {
        scenario_id: {ds_id: xarray.open_dataset(res_path_dict[1.0]) for ds_id, res_path_dict in ds_dict.items()} for scenario_id, ds_dict in data_path_dict[current_period].items()
    } for current_period in [select_historic, select_future]
}

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(16,30))

ax1 = fig1.add_subplot(3,1,1 ,projection=cartopy.crs.PlateCarree(),)
data_dict[select_historic][historic_scenario_str]['climate_zone']['kg_class'].plot.contourf(ax=ax1, 
                                                                                            transform=cartopy.crs.PlateCarree(),
                                                                                            cbar_kwargs={"location": "bottom"},
                                                                                           ) 
ax1.coastlines()
ax1.set_title(f'KG CLimate Zones {select_historic}')

ax1 = fig1.add_subplot(3,1,2 ,projection=cartopy.crs.PlateCarree(),)
data_dict[select_future][select_future_scenario]['climate_zone']['kg_class'].plot.contourf(ax=ax1, 
                                                                                           transform=cartopy.crs.PlateCarree(),
                                                                                           cbar_kwargs={"location": "bottom"},
                                                                                          )
ax1.coastlines()
ax1.set_title(f'KG CLimate Zones {select_future}')

ax1 = fig1.add_subplot(3,1,3 ,projection=cartopy.crs.PlateCarree(),)
diff_arr = data_dict[select_future][select_future_scenario]['climate_zone']['kg_class'] != data_dict[select_historic][historic_scenario_str]['climate_zone']['kg_class']
diff_arr.plot.contourf(ax=ax1, 
                       transform=cartopy.crs.PlateCarree(),
                       cbar_kwargs={"location": "bottom"},
                      )
ax1.coastlines()
ax1.set_title(f'KG Climate Zones diff {select_future} compared to {select_historic}')


Paired with the climate zones maps, we have the mean and standard deivation for each month of the year, averaged over each climate period covered by teh dataset.

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(3*16,12*10))

for month in range(12):
    ax1 = fig1.add_subplot(12,2,month*2 +1 ,projection=cartopy.crs.PlateCarree())
    plot_ds = data_dict[select_historic][historic_scenario_str]['climate_mean'].loc[{'time': month+1}]['precipitation']
    plot_ds.plot.contourf(ax=ax1, transform=cartopy.crs.PlateCarree()) 

    ax1 = fig1.add_subplot(12,2,month*2 +2 ,projection=cartopy.crs.PlateCarree())
    plot_ds = data_dict[select_historic][historic_scenario_str]['climate_mean'].loc[{'time': month+1}]['air_temperature']
    plot_ds.plot.contourf(ax=ax1, transform=cartopy.crs.PlateCarree()) 


Since the climate zones represent groupings of similar climates, we should be able to learn the mapping from the climate means and standard deviations, to a climate classification. We can experiment with different combinations of inputs to see which climate info is most important for deterjining climate classifications. In addition we can make the problem slightly simpler by focussing on a smaller group of classes, of wich there are only 5, rather than the full 30 subgroups on the full classification scheme.

## Load tabular data
In order to train a machine learning algorithm, we need lots of example to show to the algorthm so it can learn the mapping from predictors to target. If we use the whole climate zones map, we only have a few example (19 from all periods) to learn from. So instead we are going to make prediction for each land point in the data set independently. So insteadof using the gridded data directly, we have transformed the data into a tabular dataset (see the Data Prep notebook for details of how this was done). In this form, the different points (by sptial location and time period) are represented in rows, and the different climate attributes for each point and the corresponing climate zone for each point is represented as different columns or features.

This sort of data preparation to make data suitable for use with ML is a very common task. For some datasets there are only a few small tweaks required to be able to use the data, in other cases a lot of transforming and cleaning is required for the data to be suitable. The less work required to use a dataset with ML, the more it can be described as an ML-ready dataset. We will discuss more on the concept of ML ready data (some times referred to as AI ready data). Throughout the tutorial.

### Load data using pandas

In [ ]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

In [ ]:
zones_df = pandas.read_csv(mlready_data_path)

In [ ]:
zones_df.columns

### Exploring data distributions
A key part of Exploratory Data Analsyis is to under key atrributes of the data, such as
* range of each feature
* distribution of values
* correlations between different variables, especially between predictors and target variables.
* coverage or representivity of the dataset. By this mean to what extent are the interesting relationship or phenomena that we want to model preent in the dataset for the algorithm to learn from?

In [ ]:
zones_df.columns

In [ ]:
zones_df['climate_subgroup'].value_counts().plot.bar(figsize=(8,5))

We can see here that we have **class imbalance** in our data. 
In classification problems, like this one, the term class imbalance is defined as follows:

An imbalanced classification problem is an example of a classification problem where the distribution of examples across the known classes is biased or skewed

(taken from article [*A Gentle Introduction to Imbalanced Classification by Jason Brownlee*](https://machinelearningmastery.com/what-is-imbalanced-classification/) )

Lets look at the distribution of our target variable, in this case it is the rotors column of the data. This variables labels whether or not one or more rotors occured in a given 3 hour period.

A class imbalance is where the there is not an approximately equal fraction of the data points that belong to different target classes. The more the imbalance the greater the problem. The reason this is a problem for machine learning is that the loss function which informs the iterative updating of parameters during training, is effectively calculated over the whole training set, with all points considered euqally important. If tghere is a smaller number of points in one class, errors for that class will have a smaller impact on overall performance. For underepresented class, an algorithm can often get better scores by just "ignoring" the class altogether. This is a problem in weather and climate where extreme events are the most interesting data points, but also tend to underepresented in the training data due rarity. Our requirements usually consider such losses for data points representing extreme events to be of greater importance than other datapoints representing weather conditions close to climatological values. So where we identify a class imbalance in a dataset, we need to give careful consideration to designing different aspects of our machine learning training pipeline including:
* train/validate/test split
* selection of metrics
* analysis of predictions and errors for different classes

### Correlations between predictors and target variables
In order for a machine learning algorithm to be able to learn a mapping from our predictors (in this case monthly means and standard deivations of temperature and preciptation for particular land points on earth) to our target values (categorical classification of the climate zzone for particular land points on earth), we need 

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(25,10))
for ix_zone, zone_id in enumerate(['A','B','C','D','E']):
    ax1 = fig1.add_subplot(2,5,ix_zone+1, title=f'distribution of Janaury Air Temperature - Zone {zone_id}')
    zones_df[zones_df['climate_group'] == zone_id]['air_temperature_1.0_mean'].hist()
    ax1.set_xlim(-30,40)
    ax1 = fig1.add_subplot(2,5,ix_zone+6, title=f'distribution of July Air Temperature - Zone {zone_id}')
    zones_df[zones_df['climate_group'] == zone_id]['air_temperature_7.0_mean'].hist()
    ax1.set_xlim(-30,40)    
                           

## Exercises
We've only just scratched the surface in terms of exploring the data that we will be using for training a machine learning model.

### Explore distributions for tieme periods and scenarios
How do the climate variable distributions look different for different time periods (historical abd future) and for different climate change scenarios for the future time periods?


In [ ]:
# Insert Code

### Further Exploration of Correlations
What can we learn from looking at further correlations between predictors and targets? Are there correlations between some of the predictors, could we further reduce the number of predictors in some way to reduce the cost of training?

In [ ]:
# Insert Code

## Next steps and follow on material
- Fu9rther examples of how to work with weather and climate data - [Copernicus Climate Data Store tutorials](https://ecmwf-projects.github.io/copernicus-training-c3s/intro.html)
- Understanding how to prepare data for use with AI/ML [AI Data Readiness](https://github.com/MetOffice/ai_data_readiness)

### Data statement
This data used in this notebook is derived from the Koppen-Geiger Climate Cliassfication dataset created by GloH2O.

###     References
- [Climate Zones Dataset](https://www.gloh2o.org/koppen/#:~:text=The%20K%C3%B6ppen%2DGeiger%20climate%20classification%20maps%20are%20high%2Dresolution,Climate%20Sensitivity%20(ECS)%2C%20and%20historical%20warming%20trend)